# 06 — Diagnóstico: alinhamento patch-a-patch do SigLIP no LIBERO

Antes de considerar usar o SigLIP congelado como um "gate" espacial de
linguagem (uma 4a abordagem de fusão, sem nenhum parâmetro novo treinável,
além de Token/FiLM/CrossAttention já implementadas em
`src/act_lang/models/fusion/`), este notebook verifica com dados reais do
LIBERO se a similaridade patch-a-patch entre imagem e texto do SigLIP é
(a) espacialmente concentrada no objeto/ação certo e (b) **muda quando a
instrução muda**.

**Risco conhecido** (motivo de rodar isso antes de integrar no pipeline): o
SigLIP é treinado com uma loss contrastiva sobre um vetor **global**
(pooled), não patch a patch — literatura de dense-prediction zero-shot com
CLIP/SigLIP (MaskCLIP, GEM) mostra que a similaridade crua por patch costuma
ser ruidosa sem ajustes extras, e imagens de câmera de robô em simulação são
bem fora da distribuição de fotos web que o SigLIP viu no pré-treino. Sendo
zero-parâmetro, esse gate não tem como se corrigir via gradiente se o sinal
vier ruim — por isso vale validar antes de investir na integração completa.

**O que este notebook faz, na ordem:**
1. Clona o repo e instala as dependências (`hdf5` + `vlm`)
2. Aponta pro dataset HDF5 (baixa se ainda não tiver nesta sessão — mesmo
   caminho `/content/libero_hdf5` do `04_treino_hdf5.ipynb`, reaproveita se
   já baixado)
3. Roda `scripts/diagnose_siglip_patch_alignment.py`, que salva PNGs
   comparando o heatmap da instrução certa vs. uma instrução errada pra cada
   amostra, e imprime um resumo numérico
4. Mostra os PNGs inline

Puramente exploratório — não treina nada, não toca no modelo ACT nem em
`fusion/`. Não precisa de GPU (SigLIP congelado em modo inferência é leve —
roda até em CPU em segundos/poucos minutos para poucas amostras), mas usa se
disponível.

## 1. Repositório e ambiente

In [ ]:
!git clone -b teste https://github.com/rafaelheydt/act-lang.git
%cd act-lang

# "hdf5" traz h5py + huggingface_hub (leitura do dataset nativo); "vlm" traz
# transformers, novo, só pra este diagnóstico (SiglipModel/SiglipProcessor).
!pip install -q -e ".[hdf5,vlm]"

import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")


## 2. Dataset HDF5

Baixa só se `DATA_DIR` ainda não existir nesta sessão (reaproveita se você
já rodou o `04_treino_hdf5.ipynb` antes). `PILOT_LIMIT` pequeno é suficiente
aqui — só precisamos de poucas imagens+instruções, não do dataset inteiro.

In [ ]:
PILOT_LIMIT = 3  # None = as 40 tarefas completas (~28-32GB); aqui só precisamos de poucas amostras
DATA_DIR = "/content/libero_hdf5"  # mesmo caminho do 04_treino_hdf5.ipynb

import subprocess, sys
from pathlib import Path

if Path(DATA_DIR).is_dir() and any(Path(DATA_DIR).iterdir()):
    print(f"{DATA_DIR} já existe e não está vazio -- pulando download (reaproveitando desta sessão)")
else:
    cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR]
    if PILOT_LIMIT is not None:
        cmd += ["--limit", str(PILOT_LIMIT)]
    print("rodando:", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("
código de saída:", proc.returncode)
    assert proc.returncode == 0, "download falhou -- veja o log acima"


## 3. Diagnóstico: similaridade patch-a-patch

Pra cada uma de `N_SAMPLES` amostras (espalhadas pelo dataset, não só os
primeiros frames): roda o SigLIP congelado, calcula o heatmap de similaridade
patch-a-patch pra instrução **certa** e pra uma instrução **errada** (de
outra tarefa do dataset), salva os dois lado a lado em PNG.

Requer mais de uma tarefa distinta no `DATA_DIR` pro teste de
discriminabilidade fazer sentido (o pré-requisito `PILOT_LIMIT >= 2` acima
garante isso).

In [ ]:
N_SAMPLES = 6
OUTPUT_DIR = "siglip_patch_alignment_outputs"

import subprocess, sys
from pathlib import Path
from datetime import datetime

Path("logs").mkdir(exist_ok=True)
log_path = f"logs/siglip_diag_{datetime.now():%Y%m%d_%H%M}.log"

cmd = [
    sys.executable, "-u", "scripts/diagnose_siglip_patch_alignment.py",
    "--hdf5-dir", DATA_DIR,
    "--n-samples", str(N_SAMPLES),
    "--output-dir", OUTPUT_DIR,
]
print("rodando:", " ".join(cmd))
print("log em:", log_path, "
" + "=" * 60)

with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("
" + "=" * 60)
print(f"processo encerrado com código {proc.returncode}"
      + (" -- verifique o log acima" if proc.returncode != 0 else " -- ok"))


## 4. Heatmaps

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

for path in sorted(glob.glob(f"{OUTPUT_DIR}/*.png")):
    plt.figure(figsize=(13, 4.5))
    plt.imshow(Image.open(path))
    plt.axis("off")
    plt.title(path)
    plt.show()


## 5. Como interpretar

Olhe tanto os heatmaps acima quanto o resumo numérico impresso pela célula
da seção 3 (`diff médio`):

- **Heatmap da instrução certa concentrado no objeto/região certa, e
  visivelmente diferente do heatmap da instrução errada** → o sinal existe e
  é específico da linguagem; vale a pena desenhar a integração completa no
  pipeline (um módulo que substitui `VisionBackbone` + `fuse()`, já que o
  produto escalar só faz sentido no espaço nativo do SigLIP).
- **Heatmaps parecidos entre instrução certa/errada, ou `diff médio` perto
  de zero** → confirma o risco: o sinal é saliência genérica, não
  alinhamento com a linguagem. O gate zero-parâmetro provavelmente não vale
  a pena sem alguma forma de calibração/fine-tuning — reconsiderar as outras
  opções discutidas (SigLIP como backbone + fusão aprendida com
  Token/FiLM/CrossAttention, ou um VLM totalmente fundido tipo PaliGemma).